In [1]:
import numpy as np
import pandas as pd

# Definirea datelor

In [2]:
columns = ['loves_popcorn', 'loves_soda', 'age', 'cold_as_ice(target)']
data = pd.read_csv('example-dataset.csv',skiprows=1, header=None, names=columns)
data

,loves_popcorn,loves_soda,age,cold_as_ice(target)
0,1,1,7,0
1,1,0,12,0
2,0,1,18,1
3,0,1,35,1
4,1,1,38,1
5,1,0,50,0
6,0,0,83,0


# Clasa NODE

In [3]:
class Node:
    def __init__(self, index_conditie=None, valoare_conditie=None, nod_stang=None, nod_drept=None, gini_impurity=None, valoare_finala=None):
        self.index_conditie = index_conditie
        self.valoare_conditie = valoare_conditie
        self.nod_stang = nod_stang
        self.nod_drept = nod_drept
        self.gini_impurity = gini_impurity

        self.valoare_finala = valoare_finala
        

# Clasa DECISION TREE

    # pentru gini impurity
        # luam fiecare linie pentru o coloana
        # vedem care inregistrari satisfac conditia si care nu :
            # conditie satisfacuta: impartim cele care au rez. pozitiv si care au rezultat negativ
        # calculam gini impurity
        # gini impurity = 1 - (probabilatea rasp pozitiv)\*\*2 - (probabilitatea rasp negativ)\*\*2
            # probabilitatea rasp pozitiv/negativ = nr de entitati cu raspuns pozitiv/negativ / nr total entitatilor ce satisfac conditia 
            
        # se calculeaza Total gini impurity 
            # weight = 
            # (nr entitati ce satsifac conditia / nr total entitatilor) * gini impurity (celor ce satisfac conditia) 
            # + 
            # (nr entitati ce nu satisfac conditia / nr total entitatilor) * gini impurity (celor ce nu satisfac conditia)
            
        # pentru cazul cand avem numere
            # se sorteaza numerele pe coloana in ordine crescatoare
            # se calculeaza valoarea medie pentru entitatile adiacente (2 entitati vecine)
            # se calculeaza Total gini impurity pentru fiecare valoare medie in parte
            # se alege cel mai mic Total gini impurity dintre cele calculate
            
        # se alege cel mai mic Total gini impurity si se pune ca conditite

In [4]:
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=2):
        self.root = None

        # conditiile de oprire
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

    def build_tree(self, dataset, curr_depth=0):

        X,Y = dataset[:,:-1], dataset[:,-1]
        num_samples, num_features = np.shape(X)

        if num_samples >= self.min_samples_split and curr_depth <= self.max_depth:

            best_condition = self.search_best_condition(dataset, num_samples, num_features)

            if best_condition['total_gini_impurity'] >= 0:

                left_subtree = self.build_tree(best_condition['dataset_left'], curr_depth + 1)

                right_subtree = self.build_tree(best_condition['dataset_right'], curr_depth + 1)

                return Node(best_condition['feature_index'],best_condition['threshold'], left_subtree, right_subtree, best_condition['total_gini_impurity'])
        
        leaf_value = self.calculate_leaf_value(Y)
        
        return Node(valoare_finala=leaf_value)

    def search_best_condition(self, dataset, num_samples, num_features):

        best_condition = {}
        best_total_gini_impurity = float("inf")

        for feature_index in range(num_features): 
            feature_values = dataset[:,feature_index] 
            possible_threshold = np.unique(feature_values) 
            
            possible_threshold.sort()
                
            for threshold in possible_threshold: 

                dataset_left, dataset_right = self.split(dataset, feature_index, threshold)
                
                if len(dataset_left) > 0 and len(dataset_right) > 0 :

                    y, left_y, right_y = dataset[:,-1], dataset_left[:,-1], dataset_right[:,-1] 

                    curr_total_gini_impurity = self.total_gini_impurity(y, left_y, right_y)

                    if curr_total_gini_impurity < best_total_gini_impurity :
                        best_condition['feature_index'] = feature_index
                        best_condition['threshold'] = threshold
                        best_condition['dataset_left'] = dataset_left
                        best_condition['dataset_right'] = dataset_right
                        best_condition['total_gini_impurity'] = curr_total_gini_impurity
                        best_total_gini_impurity = curr_total_gini_impurity
            
        return best_condition
    
    def split(self, dataset, feature_index, threshold):
        dataset_left = np.array([row for row in dataset if row[feature_index] <= threshold])
        dataset_right = np.array([row for row in dataset if row[feature_index] > threshold])

        return dataset_left,dataset_right
    
    def total_gini_impurity(self, parent, l_child, r_child): 
        weight_l = len(l_child)/len(parent)
        weight_r = len(r_child)/len(parent)
        gini_value = weight_l * self.gini_impurity(l_child) + weight_r * self.gini_impurity(r_child)
        return gini_value
    
    def gini_impurity(self, y):
        gini = 0
        class_labels = np.unique(y)
        for clasa in class_labels:
            sum_true = 0
            for enty in y:
                if enty == clasa:
                    sum_true += 1
            
            p_cls = sum_true / len(y)
            gini += p_cls**2
        return 1 - gini
    
    def calculate_leaf_value(self, Y):
        Y = list(Y)
        return max(Y, key=Y.count)
    
    def print_tree(self, tree=None, indent=" "):
        if not tree:
            tree = self.root
        
        if tree.valoare_finala is not None:
            print(tree.valoare_finala)
        else:
            print("X_"+str(columns[tree.index_conditie]), "<=", tree.valoare_conditie, "?", tree.gini_impurity)
            print("%sleft:" % (indent), end="")
            self.print_tree(tree.nod_stang, indent + indent)
            print("%sright:" % (indent), end="")
            self.print_tree(tree.nod_drept, indent + indent)

    def fit(self, X, Y):
        dataset = np.concatenate((X, Y), axis=1)
        self.root = self.build_tree(dataset)

    def predict(self, X):
        predictions = [self.make_prediction(x, self.root) for x in X]
        return predictions
    
    def make_prediction(self, x, tree):
        if tree.valoare_finala != None: return tree.valoare_finala
        feature_val = x[tree.index_conditie]
        if feature_val <= tree.valoare_conditie:
            return self.make_prediction(x, tree.nod_stang)
        else:
            return self.make_prediction(x, tree.nod_drept)


# "Antrenarea"

In [5]:
X = data.iloc[:, :-1].values
Y = data.iloc[:, -1].values.reshape(-1,1)

dec_tree = DecisionTree(min_samples_split=3,max_depth=3)
dec_tree.fit(X, Y)
dec_tree.print_tree()

X_loves_soda <= 0 ? 0.21428571428571427
 left:X_loves_popcorn <= 0 ? 0.0
  left:0
  right:0
 right:X_age <= 7 ? 0.0
  left:0
  right:X_loves_popcorn <= 0 ? 0.0
    left:1
    right:1


In [6]:
Y_pred = dec_tree.predict(X)
from sklearn.metrics import accuracy_score
accuracy_score(Y, Y_pred)

1.0

# Al doilea dataset

In [15]:
columns = ['sepal_lenght', 'sepal_width', 'petal_lenght', 'petal_width', 'type']
data = pd.read_csv("Iris.csv", skiprows=1, header=None, names=columns)
data.head(5)

,sepal_lenght,sepal_width,petal_lenght,petal_width,type
1,5.1,3.5,1.4,0.2,0
2,4.9,3.0,1.4,0.2,0
3,4.7,3.2,1.3,0.2,0
4,4.6,3.1,1.5,0.2,0
5,5.0,3.6,1.4,0.2,0


In [16]:
X = data.iloc[:, :-1].values
Y = data.iloc[:, -1].values.reshape(-1,1)
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=41)

In [17]:
dec_tree = DecisionTree(min_samples_split=3,max_depth=3)
dec_tree.fit(X_train, Y_train)
dec_tree.print_tree()

X_petal_lenght <= 1.9 ? 0.3291139240506329
 left:X_sepal_lenght <= 4.3 ? 0.0
  left:0.0
  right:X_sepal_lenght <= 4.4 ? 0.0
    left:0.0
    right:X_sepal_lenght <= 4.6 ? 0.0
        left:0.0
        right:0.0
 right:X_petal_width <= 1.5 ? 0.07281324645358371
  left:X_petal_lenght <= 4.9 ? 0.0
    left:X_sepal_lenght <= 4.9 ? 0.0
        left:1.0
        right:1.0
    right:2.0
  right:X_petal_lenght <= 5.0 ? 0.07317073170731708
    left:X_sepal_width <= 2.8 ? 0.16666666666666666
        left:2.0
        right:1.0
    right:X_sepal_lenght <= 5.8 ? 0.0
        left:2.0
        right:2.0


In [18]:
Y_pred = dec_tree.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy_score(Y_test, Y_pred)

0.9333333333333333

# Al treilea dataset

In [19]:
columns = ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']
data = pd.read_csv(r"C:\Users\colit\Desktop\Master 1\Metode Avansate de Analiza a Datelor\Proiect - Random Forest\Dataset\winequality-red-fixed-csv.csv", skiprows=1, header=None, names=columns)
data.head(5)

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11,34,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25,67,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15,54,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17,60,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11,34,0.9978,3.51,0.56,9.4,5


In [20]:
X = data.iloc[:, :-1].values
Y = data.iloc[:, -1].values.reshape(-1,1)
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=41)

In [27]:
dec_tree = DecisionTree(min_samples_split=11,max_depth=11)
dec_tree.fit(X, Y)
dec_tree.print_tree()

X_alcohol <= 10.2 ? 0.5831646729334011
 left:X_sulphates <= 0.57 ? 0.499731787137058
  left:X_total_sulfur_dioxide <= 98.0 ? 0.39968508400928515
    left:X_alcohol <= 9.7 ? 0.439911779954162
        left:X_alcohol <= 9.05 ? 0.367447369299036
                left:X_fixed_acidity <= 7.4 ? 0.4619047619047619
                                left:4.0
                                right:6.0
                right:X_pH <= 3.53 ? 0.32563286015005133
                                left:X_residual_sugar <= 4.3 ? 0.316289016679541
                                                                left:X_volatile_acidity <= 0.23 ? 0.2927572985538749
                                                                                                                                left:6.0
                                                                                                                                right:X_volatile_acidity <= 0.965 ? 0.2810313075506446
                                   

In [28]:
Y_pred = dec_tree.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy_score(Y_test, Y_pred)

0.8675